# GPC molecular-weight calculation (Mn, Mw, Đ)

This notebook calculates the **number-average (Mn)** and **weight-average (Mw)** molecular
weights and the **dispersity (Đ = Mw / Mn)** of a *single, already-deconvoluted* GPC peak.

- By default it runs on the bundled demo file **`sample_GPC_peak.csv`** (synthetic data, not real measurements), so anyone can reproduce the workflow out of the box.
- To analyse your own data, set `file_path` to your CSV (two columns: `Time`, `Signal`) and update the calibration constants `a, b, c, d` for your column/standards.

See the README for input format, the calibration equation, and method notes.

In [ ]:
# ===== Block 1: read data + plot the raw peak =====
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline

# ---- Only edit these three lines for your own data ----
file_path  = 'sample_GPC_peak.csv'   # path to your CSV (default = bundled demo data)
time_col   = 'Time'                   # name of the retention-time column
signal_col = 'Signal'                 # name of the detector-signal column
# -------------------------------------------------------

data = pd.read_csv(file_path)
print(data.head())                    # confirm the column names are correct

time   = data[time_col].values
signal = data[signal_col].values

plt.figure(figsize=(10, 4))
plt.plot(time, signal, 'b-', label='GPC peak')
plt.xlabel('Retention time (min)')
plt.ylabel('Signal intensity')
plt.title('Raw GPC peak')
plt.grid(True)
plt.legend()
plt.show()

In [ ]:
# ===== Block 2: calculate Mw, Mn, and dispersity =====

# ---- Calibration curve: log10(M) = a + b*t + c*t**2 + d*t**3 ----
# NOTE: these constants are specific to one column / one set of standards.
# Replace them with the values from your own calibration.
a = 20.2594
b = -1.5829
c = 0.05776
d = -0.000794689

# ---- Peak start/end read off the plot above (edit for your peak) ----
t_start = 28.0    # peak start time
t_end   = 32.0    # peak end time
dt      = 0.05    # slice spacing (smaller = finer)
# --------------------------------------------------------------------

def calc_M(t):
    logM = a + b*t + c*t**2 + d*t**3
    return 10**logM

t_slices      = np.arange(t_start, t_end, dt)
M_slices      = calc_M(t_slices)
signal_slices = np.interp(t_slices, time, signal)

# Core calculation (GPC concentration signal is proportional to mass w):
#   Mw = sum(w*M) / sum(w)
#   Mn = sum(w)   / sum(w/M)
Mw  = np.sum(signal_slices * M_slices) / np.sum(signal_slices)
Mn  = np.sum(signal_slices) / np.sum(signal_slices / M_slices)
PDI = Mw / Mn   # dispersity, Đ

print(f"Mw  = {Mw:,.0f}")
print(f"Mn  = {Mn:,.0f}")
print(f"PDI = {PDI:.3f}")

# ---- Verification plot: do the slice points cover the whole peak? ----
plt.figure(figsize=(8, 4))
plt.plot(time, signal, 'b-', label='raw peak')
plt.scatter(t_slices, signal_slices, color='red', s=10, label='slice points')
plt.axvline(t_start, color='gray', ls='--')
plt.axvline(t_end,   color='gray', ls='--')
plt.xlabel('Retention time (min)')
plt.ylabel('Signal intensity')
plt.title('Peak and integration range')
plt.legend()
plt.show()